In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## dataset corpus extraction

### finance bench

In [3]:
!pip install -q datasets pdfplumber

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 23.4 MB/s eta 0:00:00


In [ ]:
import requests
import pdfplumber
import io
import json
from datasets import load_dataset

# 2. Load the dataset
print("Loading dataset...")
dataset = load_dataset("PatronusAI/financebench", split="train")

# Filter for "Information Extraction" category
filtered_data = dataset.filter(lambda example: example['question_reasoning'] == 'Information extraction')

print(f"Found {len(filtered_data)} Information Extraction questions.")


Loading dataset...


README.md: 0.00B [00:00, ?B/s]

financebench_merged.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/150 [00:00<?, ? examples/s]

Filter:   0%|          | 0/150 [00:00<?, ? examples/s]

Found 31 Information Extraction questions.


In [ ]:
# 3. Create Unique Document Map
unique_docs = {}
for row in filtered_data:
    doc_id = row['doc_name']
    doc_url = row['doc_link']
    if doc_id not in unique_docs:
        unique_docs[doc_id] = doc_url

print(f"Found {len(unique_docs)} unique documents to process.")

# Define Headers to look like a browser
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

Found 24 unique documents to process.


In [ ]:
def get_pdf_text(url):
    try:
        # Add headers and timeout
        response = requests.get(url, headers=HEADERS, timeout=15)
        response.raise_for_status()

        with pdfplumber.open(io.BytesIO(response.content)) as pdf:
            text = ""
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
        return text
    except Exception as e:
        print(f"Warning: Could not download {url}. Error: {e}")
        return None

In [ ]:
# 4. Processing and Saving to File
output_filename = "finance_bench_corpus.json"
processed_count = 0

print(f"Starting download. Saving to {output_filename}...")

# We will build the list and then dump it to JSON
final_corpus = []

for doc_name, url in unique_docs.items():
    print(f"Downloading: {doc_name}...", end=" ")

    text = get_pdf_text(url)

    if text:
        final_corpus.append({
            "doc_name": doc_name,
            "text": text,
            "url": url
        })
        print("Success ✅")
        processed_count += 1
    else:
        print("Failed ❌")

#Save to a file immediately
with open(output_filename, "w", encoding="utf-8") as f:
    json.dump(final_corpus, f, indent=4)

print("\n" + "="*40)
print(f"COMPLETED. Successfully saved {processed_count} documents.")
print(f"You can now download '{output_filename}' from the files tab on the left.")


Starting download. Saving to finance_bench_corpus.json...
Downloading: 3M_2018_10K... 

KeyboardInterrupt: 

save file

In [ ]:
import shutil

# Copy the file there
shutil.copy("finance_bench_corpus.json", "/content/drive/MyDrive/finance_bench_corpus.json")

print("Saved to your Google Drive!")

Saved to your Google Drive!


load file

In [ ]:
import json

# Load the file back
with open("/content/drive/MyDrive/finance_bench_corpus.json", "r") as f:
  data = json.load(f)


# Print the first 500 characters of the first document
print(f"Total documents: {len(data)}")
print("--- START OF TEXT PREVIEW ---")
print(data[0]['text'][:500])
print("--- END OF TEXT PREVIEW ---")

Total documents: 24
--- START OF TEXT PREVIEW ---
low
UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
FORM 10-K
☒ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE
SECURITIES EXCHANGE ACT OF 1934
For the fiscal year ended December 31, 2018
Commission file number 1-3285
3M COMPANY
State of Incorporation: Delaware I.R.S. Employer Identification No. 41-0417775
Principal executive offices: 3M Center, St. Paul, Minnesota 55144
Telephone number: (651) 733-1110
SECURITIES REGISTERED PURSUANT TO SECTION 12(b) OF THE ACT:
Name
--- END OF TEXT PREVIEW ---


extract the failed ones

In [ ]:
import json
import pdfplumber
import os

# --- CONFIGURATION ---
# Format: "DOC_ID": {"filename": "YOUR_UPLOADED_FILENAME.pdf", "url": "ORIGINAL_URL"}
manual_map = {
    "MICROSOFT_2016_10K": {
        "filename": "Microsoft_2016_10K.pdf",
        "url": "https://microsoft.gcs-web.com/static-files/a779c4f1-d788-4890-83fb-633d198efe7e"
    },
    "PEPSICO_2021_10K": {
        "filename": "/content/PEPSICO_2021_10K.pdf",
        "url": "https://pepsico.gcs-web.com/static-files/a5a1d988-8e28-4dc7-ac4e-e6a2abfd0310"
    },
    "PEPSICO_2022_10K": {
        "filename": "PEPSICO_2022_10K.pdf",
        "url": "https://pepsico.gcs-web.com/static-files/d051bdd6-c6d2-4814-826f-ece589b88d4c"
    }
}

EXISTING_FILE = "/content/drive/MyDrive/finance_bench_corpus.json"

# 1. Load existing data
try:
    with open(EXISTING_FILE, "r") as f:
        existing_data = json.load(f)
    print(f"Current document count: {len(existing_data)}")

    # Create a check to ensure we don't add duplicates if you run this twice
    existing_ids = {doc['doc_name'] for doc in existing_data}

except FileNotFoundError:
    print(f"Error: {EXISTING_FILE} not found. Make sure you ran the previous download step.")
    existing_data = []
    existing_ids = set()

# 2. Process Manual Files
added_count = 0

for doc_name, info in manual_map.items():
    filename = info['filename']
    original_url = info['url']

    # Skip if already in the dataset
    if doc_name in existing_ids:
        print(f"Skipping {doc_name}: Already exists in JSON.")
        continue

    # Skip if file wasn't uploaded
    if not os.path.exists(filename):
        print(f"Skipping {doc_name}: File '{filename}' not found in Colab files.")
        continue

    print(f"Processing local file: {filename}...", end=" ")

    try:
        with pdfplumber.open(filename) as pdf:
            text = ""
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"

        # Add to the list with the CORRECT URL
        existing_data.append({
            "doc_name": doc_name,
            "text": text,
            "url": original_url
        })
        print("Success ✅")
        added_count += 1

    except Exception as e:
        print(f"Failed ❌ ({e})")

# 3. Save Updated File
if added_count > 0:
    with open(EXISTING_FILE, "w", encoding="utf-8") as f:
        json.dump(existing_data, f, indent=4)
    print(f"\nDONE. Added {added_count} documents.")
    print(f"New total count: {len(existing_data)}")
else:
    print("\nNo new documents were added.")

Current document count: 24
Skipping MICROSOFT_2016_10K: Already exists in JSON.
Skipping PEPSICO_2021_10K: Already exists in JSON.
Skipping PEPSICO_2022_10K: Already exists in JSON.

No new documents were added.


### simpleWikiQA

Simple WikiQA is a specific subset created by the authors from the larger SimpleQA benchmark. We will load the SimpleQA benchmark and extract the wikipedia articles from each question.

In [4]:
from datasets import load_dataset
import pandas as pd

# Load the SimpleQA dataset
print("Loading SimpleQA dataset...")
dataset = load_dataset("basicv8vc/SimpleQA", split="test")



Loading SimpleQA dataset...


README.md:   0%|          | 0.00/506 [00:00<?, ?B/s]

simple_qa_test_set.csv: 0.00B [00:00, ?B/s]

Generating test split:   0%|          | 0/4326 [00:00<?, ? examples/s]

In [5]:
import ast

# Extract Unique Wikipedia URLs
print("Extracting Wikipedia URLs from metadata...")

unique_urls = set()
wiki_questions_count = 0

for row in dataset:

    meta_string = row.get('metadata')

    # If it's empty, skip
    if not meta_string:
        continue

    try:

        meta_dict = ast.literal_eval(meta_string)
    except (ValueError, SyntaxError):
        continue

    if 'urls' not in meta_dict:
        continue

    url_list = meta_dict['urls']
    found_wiki_in_this_row = False

    # Iterate through the list of URLs
    for url_entry in url_list:
        # Handle the squashed URLs separated by newline \n
        potential_urls = url_entry.split('\n')

        for clean_url in potential_urls:
            clean_url = clean_url.strip()

            # Filter for Wikipedia
            if "en.wikipedia.org" in clean_url:
                unique_urls.add(clean_url)
                found_wiki_in_this_row = True

    if found_wiki_in_this_row:
        wiki_questions_count += 1

print("-" * 30)
print(f"Total Questions with Wiki Links: {wiki_questions_count}")
print(f"Total Unique Wikipedia Documents found: {len(unique_urls)}")
print("-" * 30)

# 3. Save to file
output_file = "simple_wiki_urls.txt"
with open(output_file, "w") as f:
    for url in unique_urls:
        f.write(url + "\n")

print(f"Saved unique URLs to '{output_file}'.")

Extracting Wikipedia URLs from metadata...
------------------------------
Total Questions with Wiki Links: 3449
Total Unique Wikipedia Documents found: 4087
------------------------------
Saved unique URLs to 'simple_wiki_urls.txt'.


scrape the text from the Wikipedia links

In [6]:
!pip install -q beautifulsoup4 requests tqdm

In [7]:
import requests
from bs4 import BeautifulSoup
import json
import tqdm
import time
import random

# --- CONFIGURATION ---
INPUT_FILE = "/content/simple_wiki_urls.txt"
OUTPUT_FILE = "simple_wiki_corpus.json"

# Wikipedia requires a User-Agent to avoid blocking
HEADERS = {
    "User-Agent": "MyActiveReadingReproduction/1.0 (dimitris.kurtis13@gmail.com) based on python requests"
}

# 1. Load URLs
with open(INPUT_FILE, "r") as f:
    urls = [line.strip() for line in f.readlines()]

print(f"Loaded {len(urls)} URLs to scrape.")

# 2. Scrape Function
def scrape_wiki_text(url):
    try:
        response = requests.get(url, headers=HEADERS, timeout=10)

        if response.status_code != 200:
            return None

        soup = BeautifulSoup(response.content, 'html.parser')

        # Wikipedia stores the main article text in 'div' with id 'mw-content-text'
        content_div = soup.find(id="mw-content-text")
        if not content_div:
            return None

        # Extract all paragraphs <p>
        paragraphs = content_div.find_all('p')

        # Combine text
        clean_text = "\n".join([p.get_text() for p in paragraphs])

        # Get Title (for metadata)
        title = soup.find(id="firstHeading")
        title_text = title.get_text() if title else "Unknown"

        return {"title": title_text, "text": clean_text}

    except Exception:
        return None

# 3. Main Loop
corpus_data = []
errors = 0

print("Starting Scrape... (This might take a while)")

# Use tqdm for a progress bar
for url in tqdm.tqdm(urls):
    data = scrape_wiki_text(url)

    if data and len(data['text']) > 100: # Filter out empty pages
        corpus_data.append({
            "doc_name": data['title'],
            "url": url,
            "text": data['text']
        })
    else:
        errors += 1

    # Be polite to Wikipedia servers (small random sleep)
    time.sleep(random.uniform(0.1, 0.3))

# 4. Save to JSON
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(corpus_data, f, indent=4)

print("\n" + "="*30)
print(f"Scraping Complete.")
print(f"Successfully scraped: {len(corpus_data)}")
print(f"Failed/Empty: {errors}")
print(f"Saved to {OUTPUT_FILE}")

Loaded 4087 URLs to scrape.
Starting Scrape... (This might take a while)


100%|██████████| 4087/4087 [54:01<00:00,  1.26it/s]



Scraping Complete.
Successfully scraped: 4020
Failed/Empty: 67
Saved to simple_wiki_corpus.json
